<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex5/ex5_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 5: LLM Prompting and Prompt Engineering Part 2

In part 2, we experiment with prompting instruction-tuned Large Language Models (LLMs), and evaluate their performance on a linguistic annotation task involving structured outputs.

The goal of this assignment is to gain some experience working with instruction-tuned LLMs. To this end, you will learn how to

- query an instruction-tuned LLM with default chat templates (`Llama-3.2-3B-Instruct`)
- parse LLM outputs for structured responses using JSON and `Pydantic`
- implement error handling for edge cases where the model fails to output the expected data format.
- compare the instruction-tuned model with the base model from part 1 on the same task.

The task we use for this purpose is a simple Tokenization and Part-of-Speech tagging task using data taken from Universal Dependencies.

To facilitate working with LLMs, we will again use the Unsloth library. Note that Unsloth provides both freeware and closed-source proprietary software. For our purposes, the freeware is sufficient! For more information on Unsloth, see the docs [here](https://github.com/unslothai/unsloth).

This notebook is adapted from [this example](https://colab.research.google.com/drive/1T5-zKWM_5OD21QHwXHiV9ixTRR7k3iB9?usp=sharing) by Unsloth.


Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

### NOTE: Expected execution times
We have provided expected execution times throughout the notebook as a guide. These are intended to be approximate, but should give you some idea for what to expect. If your runtimes far exceed these expected execution times, you may want to consider modifying your approach. These are denoted with ⌛ .

### NOTE: GPU Usage
It is expected that you load the model onto a GPU for inference. For other parts of the code, such as data preparation, a GPU is not necessary. To avoid waiting for resources unnecessarily, we recommend doing as much as you can on a CPU instance and change the runtime type as necessary. We've highlighted the cells that need a GPU with ⚡

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

## 1) Installing dependencies

In [1]:
%%capture
!pip install levenshtein
!pip install unsloth==2026.11.4 unsloth_zoo==2026.11.4

In [2]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. This exercise needs an NVIDIA GPU: in Colab, select one at 'Runtime' → 'Change runtime type'.")

GPU: Tesla T4


## 2) Model Loading

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# Note, here we specify the instruction-tuned version of Llama-3.2-3B
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


FastLanguageModel.for_inference(model) # Enable native 2x faster inference

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1126 14:31:11.424000 442 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 14:31:11.559000 442 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNo

## 3) Data Loading and Preparation

In [4]:
# load data
import random
import pandas as pd
from transformers import set_seed

seed = 42

set_seed(seed) # seeds Python, NumPy and PyTorch (CPU and GPU)

dataset_url = "https://raw.githubusercontent.com/tannonk/prompting_exercise/refs/heads/main/data/en_ewt-ud-dev-pos.json"
df = pd.read_json(dataset_url, lines=True)

# For each input sentence, we'll build the target as a list of dictionaries containing keys for the token and pos tag. This is what we want our LLM annotator to predict.
df['target'] = df.apply(lambda x: [{'token': token, 'pos': pos} for token, pos in zip(x['tokens'].split(), x['upos'].split())], axis=1)

# We'll sample 100 items for testing purposes
test_data = df.sample(n=100, random_state=seed)
train_data = df.drop(test_data.index)

print(f"Train data: {len(train_data)}")
print(f"Test data: {len(test_data)}")

test_data.head()


Train data: 1408
Test data: 100


,sentence,tokens,upos,xpos,target
578,"""...there is no companion quite so devoted, so...",""" ... there is no companion quite so devoted ,...",PUNCT PUNCT PRON VERB DET NOUN ADV ADV ADJ PUN...,"`` , EX VBZ DT NN RB RB JJ , RB JJ , RB JJ CC ...","[{'token': '""', 'pos': 'PUNCT'}, {'token': '....."
1146,"Great computer repair store, highly recommended.","Great computer repair store , highly recommend...",ADJ NOUN NOUN NOUN PUNCT ADV VERB PUNCT,"JJ NN NN NN , RB VBN .","[{'token': 'Great', 'pos': 'ADJ'}, {'token': '..."
382,You wear your heart on your sleeve ... and sin...,You wear your heart on your sleeve ... and sin...,PRON VERB PRON NOUN ADP PRON NOUN PUNCT CCONJ ...,"PRP VBP PRP$ NN IN PRP$ NN , CC IN PRP VBP DT ...","[{'token': 'You', 'pos': 'PRON'}, {'token': 'w..."
583,for Books that Speak for Themselves....,for Books that Speak for Themselves ....,ADP NOUN PRON VERB ADP PRON PUNCT,"IN NNS WDT VBP IN PRP ,","[{'token': 'for', 'pos': 'ADP'}, {'token': 'Bo..."
966,yuck !!,yuck !!,INTJ PUNCT,UH .,"[{'token': 'yuck', 'pos': 'INTJ'}, {'token': '..."


### TODO: Inspect and describe the data

📝❓ What are the fields and their corresponding values in the dataframe?

📝❓ What is the difference between `upos` and `xpos`?

📝❓ What is the distribution of `upos` labels in the `test_data`?

In [5]:
from collections import Counter

print(df.iloc[0].to_dict())
print(f"\nColumns: {df.columns.tolist()}, rows: {len(df)}")

test_lengths = test_data["tokens"].str.split().str.len()
print(f"Test sentence length: mean = {test_lengths.mean():.1f}, min = {test_lengths.min()}, max = {test_lengths.max()} tokens")

differs = (test_data["sentence"].str.split() != test_data["tokens"].str.split()).sum()
print(f"Test sentences whose UD tokens differ from a whitespace split: {differs} / {len(test_data)}")

upos_counts = Counter(tag for tags in test_data["upos"] for tag in tags.split())
upos_dist = pd.DataFrame(upos_counts.most_common(), columns=["upos", "count"])
upos_dist["share"] = (upos_dist["count"] / upos_dist["count"].sum()).round(3)
upos_dist

{'sentence': 'From the AP comes this story :', 'tokens': 'From the AP comes this story :', 'upos': 'ADP DET PROPN VERB DET NOUN PUNCT', 'xpos': 'IN DT NNP VBZ DT NN :', 'target': [{'token': 'From', 'pos': 'ADP'}, {'token': 'the', 'pos': 'DET'}, {'token': 'AP', 'pos': 'PROPN'}, {'token': 'comes', 'pos': 'VERB'}, {'token': 'this', 'pos': 'DET'}, {'token': 'story', 'pos': 'NOUN'}, {'token': ':', 'pos': 'PUNCT'}]}

Columns: ['sentence', 'tokens', 'upos', 'xpos', 'target'], rows: 1508
Test sentence length: mean = 12.0, min = 2, max = 64 tokens
Test sentences whose UD tokens differ from a whitespace split: 80 / 100


,upos,count,share
0,NOUN,219,0.182
1,PUNCT,146,0.122
2,VERB,124,0.103
3,ADJ,111,0.092
4,PRON,106,0.088
5,ADP,99,0.082
6,DET,88,0.073
7,AUX,72,0.060
8,ADV,69,0.058
9,PROPN,69,0.058


### TODO: Define the basic `PromptTemplate`

Note, you can reuse the solution from part 1 of this exercise here.

In [6]:
import json

class PromptTemplate:
    def __init__(self, task_description, eos="\n\n"):
        self.task_description = task_description
        self.eos = eos

    def zero_shot_prompt(self, input_text):
        return f"{self.task_description}\n\n{input_text}\nAnswer:"

    def few_shot_prompt(self, input_text, examples=[], affix="\nAnswer:"):
        demos = [f"{ex['input']}{affix} {ex['output']}" for ex in examples]
        return "\n\n".join([self.task_description] + demos + [f"{input_text}{affix}"])

UPOS_TAGS = ["ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM",
             "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X"]

prompt_template = PromptTemplate(
    "Tokenise the sentence below and tag every token with its Universal Dependencies "
    f"part-of-speech tag (UPOS). Use only these tags: {', '.join(UPOS_TAGS)}. "
    "Split punctuation and clitics such as \"n't\" and \"'s\" into separate tokens. "
    "Return a JSON list with one object per token, with the keys \"token\" and \"pos\"."
)

demo_rows = train_data[train_data["tokens"].str.split().str.len().between(6, 15)].sample(n=3, random_state=seed)
examples = [{"input": f"Sentence: {row['sentence']}", "output": json.dumps(row["target"], ensure_ascii=False)}
            for _, row in demo_rows.iterrows()]

prompts = [prompt_template.few_shot_prompt(f"Sentence: {s}", examples) for s in test_data["sentence"]]
print(prompts[0])

Tokenise the sentence below and tag every token with its Universal Dependencies part-of-speech tag (UPOS). Use only these tags: ADJ, ADP, ADV, AUX, CCONJ, DET, INTJ, NOUN, NUM, PART, PRON, PROPN, PUNCT, SCONJ, SYM, VERB, X. Split punctuation and clitics such as "n't" and "'s" into separate tokens. Return a JSON list with one object per token, with the keys "token" and "pos".

Sentence: Like in America dinner is our main meal.
Answer: [{"token": "Like", "pos": "INTJ"}, {"token": "in", "pos": "ADP"}, {"token": "America", "pos": "PROPN"}, {"token": "dinner", "pos": "NOUN"}, {"token": "is", "pos": "AUX"}, {"token": "our", "pos": "PRON"}, {"token": "main", "pos": "ADJ"}, {"token": "meal", "pos": "NOUN"}, {"token": ".", "pos": "PUNCT"}]

Sentence: Further to our conversation, please see attached sample agreements.
Answer: [{"token": "Further", "pos": "ADV"}, {"token": "to", "pos": "ADP"}, {"token": "our", "pos": "PRON"}, {"token": "conversation", "pos": "NOUN"}, {"token": ",", "pos": "PUNCT"

## 3.2 ChatTemplates

Instruction-tuned models are typically finetuned using a predefined `ChatTemplate`.
This means that when using them for inference, it is important that we use the correct `ChatTemplate` in order to avoid "confusing" the model.

You can find more information about model `ChatTemplates` for Huggingface models [here](https://huggingface.co/docs/transformers/en/chat_templating).


In [7]:
from unsloth.chat_templates import get_chat_template

# load the chat_template from unsloth (note, the logic is similar when using native Huggingface, but here we're using Unsloth!)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # for Llama-3.1 and Llama-3.2 models
)

# Inspect the template (note, it looks more complicated than it is!)
print(tokenizer.chat_template)


{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- set date_string = "26 July 2024" %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content'] %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message + builtin tools #}
{{- "<|start_header_id|>system<|end_header_id|>

" }}
{%- if builtin_tools is defined or tools is not none %}
    {{- "Environment: ipython
" }}
{%- endif %}
{%- if builtin_tools is defined %}
    {{- "Tools: " + builtin_tools | reject('equalto', 'code_interpreter') | join(", ") + "

"}}
{%- endif %}
{{- "

### TODO: Prepare your inputs using the `ChatTemplate` for the model.

Note, you should be able to drop your custom `PromptTemplate` string into the model's default `ChatTemplate`.


In [8]:
def build_chat_prompts(prompts, system_prompt=None):
    chat_prompts = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        if system_prompt is not None:
            messages.insert(0, {"role": "system", "content": system_prompt})
        chat_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    return chat_prompts

chat_prompts = build_chat_prompts(prompts)
print(chat_prompts[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tokenise the sentence below and tag every token with its Universal Dependencies part-of-speech tag (UPOS). Use only these tags: ADJ, ADP, ADV, AUX, CCONJ, DET, INTJ, NOUN, NUM, PART, PRON, PROPN, PUNCT, SCONJ, SYM, VERB, X. Split punctuation and clitics such as "n't" and "'s" into separate tokens. Return a JSON list with one object per token, with the keys "token" and "pos".

Sentence: Like in America dinner is our main meal.
Answer: [{"token": "Like", "pos": "INTJ"}, {"token": "in", "pos": "ADP"}, {"token": "America", "pos": "PROPN"}, {"token": "dinner", "pos": "NOUN"}, {"token": "is", "pos": "AUX"}, {"token": "our", "pos": "PRON"}, {"token": "main", "pos": "ADJ"}, {"token": "meal", "pos": "NOUN"}, {"token": ".", "pos": "PUNCT"}]

Sentence: Further to our conversation, please see attached sample agreements.
Ans

## 4) Inference Pipeline

### TODO: Define a function to run inference efficiently with an LLM

Note, you can use the same inference function from part 1 of this exercise here, with one change: the prompts produced by the chat template already start with the beginning-of-sequence token `<|begin_of_text|>`, which the tokenizer would otherwise add a second time. Tokenise such prompts with `add_special_tokens=False`.

In [9]:
# Set up our inference pipeline for generation

# We'll set some default generation args that we'll pass to our inference function
# Following best practices, we'll use Pydantic class which helps with validation.
from typing import Optional
from pydantic import BaseModel

class Generation_Args(BaseModel):
    max_new_tokens: int
    temperature: float
    top_k: int
    top_p: float
    repetition_penalty: float
    do_sample: bool
    min_p: Optional[float]
    num_return_sequences: int
    use_cache: bool

# Here are some default generation args
# We use greedy decoding (do_sample = False): at each step, the model picks the most probable next token.
# This makes the outputs deterministic, so that differences in accuracy come from the prompts and not from random sampling.
# temperature, top_k, top_p and min_p only take effect if you set do_sample = True.
generation_args = Generation_Args(
    max_new_tokens = 1024, # note, for this task, we're setting the max_new_tokens to be more appropriate
    temperature = 1.0,
    top_k = 50,
    top_p = 1.0,
    repetition_penalty = 1.0,
    do_sample = False,
    use_cache = True,
    min_p = None,
    num_return_sequences = 1
)

def run_batched_inference(prompts, model, tokenizer, batch_size=10, generation_args=generation_args, add_special_tokens=False):
    """
    Runs batched inference on a list of prompts using a given model and tokenizer.

    Set the batch_size to control the number of prompts processed in each batch.
    Depending on the length of your prompts and model size the batch size may need to be adjusted.

    Args:
        prompts (list[str]): List of prompts that are passed to the model
        model (): The model used for generation
        tokenizer (): The tokenizer used for encoding and decoding the prompts
        batch_size (int): Number of prompts to run batched inference on.

    Returns:
        List[str] containing generated outputs.
    """
    tokenizer.padding_side = "left"

    outputs = []

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gen_kwargs = generation_args.model_dump(exclude_none=True)
    if not generation_args.do_sample:
        for key in ("temperature", "top_k", "top_p", "min_p"):
            gen_kwargs.pop(key, None)

    for i in range(0, len(prompts), batch_size):
        inputs = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True,
                           add_special_tokens=add_special_tokens).to(model.device)
        with torch.inference_mode():
            generated = model.generate(**inputs, **gen_kwargs, pad_token_id=tokenizer.pad_token_id)

        new_tokens = generated[:, inputs.input_ids.shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

    return outputs

## 4.1) Run Inference

### TODO: run inference!

⌛ 10-20 mins

⚡ GPU

In [10]:
import time

start = time.time()
outputs = run_batched_inference(chat_prompts, model, tokenizer)
print(f"Inference took {(time.time() - start) / 60:.1f} min\n")
print(f"Sentence: {test_data.iloc[0]['sentence']}\n\nOutput:\n{outputs[0]}")

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_

Inference took 9.7 min

Sentence: "...there is no companion quite so devoted, so communicative, so loving and so mesmerizing as a rat."

Output:
 [{"token": "there", "pos": "PRON"}, {"token": "is", "pos": "VERB"}, {"token": "no", "pos": "NOUN"}, {"token": "companion", "pos": "NOUN"}, {"token": "quite", "pos": "ADV"}, {"token": "so", "pos": "ADV"}, {"token": "devoted", "pos": "ADJ"}, {"token": "so", "pos": "ADV"}, {"token": "communicative", "pos": "ADJ"}, {"token": "so", "pos": "ADV"}, {"token": "loving", "pos": "ADJ"}, {"token": "and", "pos": "CONJ"}, {"token": "so", "pos": "ADV"}, {"token": "mesmerizing", "pos": "ADJ"}, {"token": "as", "pos": "CCONJ"}, {"token": "a", "pos": "DET"}, {"token": "rat", "pos": "NOUN"}]


## 5) Structured output validation

LLMs output text. But in practice, we often want structured data that we can process further with other automatic processes.

For this purpose, JSON is a good target data structure.


### TODO: Define a processing pipeline that extracts and validates the JSON response from the LLM.

Hint: For this you should use a combination of [`Regex`](https://www.w3schools.com/python/python_regex.asp) and [`Pydantic`](https://docs.pydantic.dev/latest/).

The output should be a valid json object with the following structure:

```json
[
    {"token": "there", "pos": "DET"}, # each dict contains a token and its corresponding POS-Tag.
    {"token": "is", "pos": "VERB"},
    {"token": "no", "pos": "ADJ"},
    {"token": "companion", "pos": "NOUN"},
    {"token": "quite", "pos": "ADV"},
    {"token": "so", "pos": "ADV"},
    {"token": "devoted", "pos": "ADV"},
    {"token": "so", "pos": "ADV"},
    ...
]
```

In [11]:
import re
from typing import List
from pydantic import BaseModel, TypeAdapter, ValidationError, field_validator

class TokenPOS(BaseModel):
    token: str
    pos: str

    @field_validator("pos")
    @classmethod
    def normalise_pos(cls, value):
        return value.strip().upper()

token_list_adapter = TypeAdapter(List[TokenPOS])

def process_output(output):
    match = re.search(r"\[\s*\{", output)
    if match is None:
        return None
    try:
        data, _ = json.JSONDecoder().raw_decode(output[match.start():])
        return [item.model_dump() for item in token_list_adapter.validate_python(data)]
    except (json.JSONDecodeError, ValidationError):
        return None

def process_outputs(outputs):
    processed = [process_output(o) for o in outputs]
    print(f"Valid outputs: {sum(p is not None for p in processed)} / {len(processed)}")
    return processed

processed_outputs = process_outputs(outputs)
print(processed_outputs[0])

Valid outputs: 98 / 100
[{'token': 'there', 'pos': 'PRON'}, {'token': 'is', 'pos': 'VERB'}, {'token': 'no', 'pos': 'NOUN'}, {'token': 'companion', 'pos': 'NOUN'}, {'token': 'quite', 'pos': 'ADV'}, {'token': 'so', 'pos': 'ADV'}, {'token': 'devoted', 'pos': 'ADJ'}, {'token': 'so', 'pos': 'ADV'}, {'token': 'communicative', 'pos': 'ADJ'}, {'token': 'so', 'pos': 'ADV'}, {'token': 'loving', 'pos': 'ADJ'}, {'token': 'and', 'pos': 'CONJ'}, {'token': 'so', 'pos': 'ADV'}, {'token': 'mesmerizing', 'pos': 'ADJ'}, {'token': 'as', 'pos': 'CCONJ'}, {'token': 'a', 'pos': 'DET'}, {'token': 'rat', 'pos': 'NOUN'}]


## 6) *Evaluation*

We align the predicted tokens with the target tokens and compute precision, recall and F1 over all test sentences, once for the tokens and once for the POS tags. A predicted token is correct if it is aligned with an identical target token; its POS tag is correct if, in addition, the tags are identical.

- **Precision** is the share of the predicted tokens that are correct, so extra tokens lower it.
- **Recall** is the share of the target tokens that are predicted correctly, so missing tokens and outputs that could not be parsed lower it.

In [12]:
# Below is some boilerplate evaluation code. You should not need to make any changes here.

import Levenshtein
import numpy as np
from typing import List, Dict, Optional

def evaluate_instance(target: List[Dict], prediction: Optional[List[Dict]]):
    """
    Compares the predicted tokens and POS tags of one sentence with the target.

    The predicted tokens are aligned with the target tokens using Levenshtein.opcodes(). A predicted token is correct
    if it is aligned with an identical target token. Its POS tag is correct if, in addition, the tags are identical.

    Args:
        target (List[Dict]): A list of dictionaries representing the target tokens and POS tags.
        prediction (List[Dict] or None): A list of dictionaries representing the predicted tokens and POS tags.

    Returns:
        dict: The number of target tokens, predicted tokens, correct tokens and correct POS tags.
    """

    target_tokens = [item['token'] for item in target]
    target_pos = [item['pos'] for item in target]

    # If there is no prediction, nothing is predicted and all target tokens are missed
    if prediction is None:
        return {'target': len(target_tokens), 'predicted': 0, 'correct tokens': 0, 'correct POS': 0}

    pred_tokens = [item['token'] for item in prediction]
    pred_pos = [item['pos'] for item in prediction]

    correct_tokens = 0
    correct_pos = 0

    # Get alignment operations between the target and predicted tokens using Levenshtein.opcodes()
    # "equal" blocks contain identical tokens; "replace", "insert" and "delete" blocks are tokenisation errors
    for tag, i1, i2, j1, j2 in Levenshtein.opcodes(target_tokens, pred_tokens):
        if tag == 'equal':
            correct_tokens += i2 - i1
            correct_pos += sum(t == p for t, p in zip(target_pos[i1:i2], pred_pos[j1:j2]))

    return {'target': len(target_tokens), 'predicted': len(pred_tokens),
            'correct tokens': correct_tokens, 'correct POS': correct_pos}

def precision_recall_f1(correct, predicted, target):
    precision = correct / predicted if predicted else 0
    recall = correct / target if target else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

def get_results(test_data: pd.DataFrame, processed_outputs: List[Optional[List[Dict]]]):
    """
    Returns precision, recall and F1 for tokenisation and POS tagging, computed over all sentences.
    The i-th output is compared with the i-th sentence of test_data.
    """
    counts = pd.DataFrame([evaluate_instance(test_data.iloc[i]['target'], processed_outputs[i])
                           for i in range(len(processed_outputs))]).sum()

    token_p, token_r, token_f1 = precision_recall_f1(counts['correct tokens'], counts['predicted'], counts['target'])
    pos_p, pos_r, pos_f1 = precision_recall_f1(counts['correct POS'], counts['predicted'], counts['target'])

    return pd.Series({'Token Precision': token_p, 'Token Recall': token_r, 'Token F1': token_f1,
                      'POS Precision': pos_p, 'POS Recall': pos_r, 'POS F1': pos_f1})

In [13]:
# To get the results, you should be able to pass your test_data DataFrame and the processed_outputs from above...
get_results(test_data, processed_outputs)

,0
Token Precision,0.201976
Token Recall,0.153333
Token F1,0.174325
POS Precision,0.162459
POS Recall,0.123333
POS F1,0.140218


## 7) Manipulating the system prompt

The system prompt is part of the `ChatTemplate` that can help to steer the model.


### TODO: Customise the system prompt for the intended task and re-run inference

Note, this is an experiment. You should try a few different system prompts and report the resulting performance in your report.


📝❓ What was the best system prompt you considered?

📝❓ Were you able to improve the performance by manipulating the system prompt? Please discuss.

⌛ 10-20 mins (per experiment run)

⚡ GPU

In [14]:
system_prompt_results = {"default (no system message)": get_results(test_data, processed_outputs)}
system_prompt_outputs = {"default (no system message)": processed_outputs}

ANNOTATOR = "You are an expert linguist who annotates English text following the Universal Dependencies guidelines."
JSON_ONLY = " Respond only with the JSON list, without any explanation or code fences."
GUIDELINES = (
    " Keep every token exactly as it appears in the sentence and do not skip, merge or correct any token."
    " Use AUX for forms of be, have, do and modal verbs when they accompany another verb,"
    " PART for infinitival 'to', 'not' and \"n't\", PROPN for names, DET for articles and demonstratives,"
    " and SCONJ for subordinating words such as 'that', 'if' and 'because' when they introduce a clause."
)
system_prompts = {
    "annotator": ANNOTATOR,
    "annotator + JSON only": ANNOTATOR + JSON_ONLY,
    "annotator + JSON only + UD rules": ANNOTATOR + JSON_ONLY + GUIDELINES,
}

for name, system_prompt in system_prompts.items():
    start = time.time()
    sp_outputs = run_batched_inference(build_chat_prompts(prompts, system_prompt), model, tokenizer)
    print(f"[{name}] {(time.time() - start) / 60:.1f} min")
    system_prompt_outputs[name] = process_outputs(sp_outputs)
    system_prompt_results[name] = get_results(test_data, system_prompt_outputs[name])

sp_table = pd.DataFrame(system_prompt_results).T
sp_table["valid outputs"] = [sum(p is not None for p in system_prompt_outputs[n]) for n in sp_table.index]
sp_table.round(3)

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t

[annotator] 10.1 min
Valid outputs: 63 / 100


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t

[annotator + JSON only] 4.6 min
Valid outputs: 95 / 100


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t

[annotator + JSON only + UD rules] 5.6 min
Valid outputs: 94 / 100


,Token Precision,Token Recall,Token F1,POS Precision,POS Recall,POS F1,valid outputs
default (no system message),0.202,0.153,0.174,0.162,0.123,0.140,98
annotator,0.278,0.137,0.183,0.210,0.103,0.139,63
annotator + JSON only,0.950,0.845,0.895,0.725,0.645,0.683,95
annotator + JSON only + UD rules,0.947,0.832,0.886,0.746,0.655,0.697,94


In [15]:
best_prompt = sp_table["POS F1"].idxmax()
confusions = Counter()
for i, prediction in enumerate(system_prompt_outputs[best_prompt]):
    if prediction is None:
        continue
    target = test_data.iloc[i]["target"]
    t_tok, p_tok = [x["token"] for x in target], [x["token"] for x in prediction]
    for tag, i1, i2, j1, j2 in Levenshtein.opcodes(t_tok, p_tok):
        if tag == "equal":
            for t, p in zip(target[i1:i2], prediction[j1:j2]):
                if t["pos"] != p["pos"]:
                    confusions[(t["pos"], p["pos"])] += 1
print(f"Best system prompt: {best_prompt}\n")
pd.DataFrame([(g, p, n) for (g, p), n in confusions.most_common(10)], columns=["gold", "predicted", "count"])

Best system prompt: annotator + JSON only + UD rules



,gold,predicted,count
0,AUX,VERB,41
1,CCONJ,CONJ,15
2,ADV,ADJ,11
3,PART,ADP,11
4,VERB,AUX,9
5,ADV,INTJ,8
6,ADP,ADV,7
7,PRON,SCONJ,7
8,NOUN,PROPN,6
9,ADJ,PROPN,6


## 8) Base model vs. instruction-tuned model

`Llama-3.2-3B-Instruct` was created from the base model `Llama-3.2-3B` that you used in part 1. The base model was trained further on many different tasks phrased as instructions together with the desired responses, formatted with the chat template (instruction tuning), and then on human preferences. Let's see what this changes for our task.

### TODO: Run the base model on the same task and compare

Load the base model and run it on the same test sentences with the prompt from your `PromptTemplate`. The base model was not trained with a chat template, so pass the prompt as plain text. Then process and evaluate the outputs with the same pipeline as above.

Note, the base model may keep generating until it reaches `max_new_tokens`. To save time, you can run this comparison on a subset of the test data (e.g. `test_data.head(20)`) and compare it with the instruction-tuned model on the same subset.

📝❓ How do the outputs of the base model differ from those of the instruction-tuned model (output format, number of valid outputs, precision, recall and F1)?

📝❓ In part 1, you finetuned the base model with LoRA on a single task. How is this different from the instruction tuning that turned `Llama-3.2-3B` into `Llama-3.2-3B-Instruct`? Use your results on the out-of-domain names from part 1 in your answer.

⌛ 5-20 mins

⚡ GPU

In [16]:
# Load the base model from part 1 under a new name, so that the instruction-tuned model stays available
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference

n_subset = 20
subset = test_data.head(n_subset)
start = time.time()
base_outputs = run_batched_inference(prompts[:n_subset], base_model, base_tokenizer, add_special_tokens=True)
print(f"Base model inference took {(time.time() - start) / 60:.1f} min\n")
print(f"First base model output:\n{base_outputs[0][:1500]}\n")
base_processed = process_outputs(base_outputs)

comparison = {
    "base model (plain prompt)": base_processed,
    "instruct, default": system_prompt_outputs["default (no system message)"][:n_subset],
    f"instruct, {best_prompt}": system_prompt_outputs[best_prompt][:n_subset],
}
comparison_table = pd.DataFrame({name: get_results(subset, outs) for name, outs in comparison.items()}).T
comparison_table["valid outputs"] = [sum(p is not None for p in outs) for outs in comparison.values()]
comparison_table.round(3)

==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base model inference took 2.3 min

First base model output:
 [{"token": "...", "pos": "PUNCT"}, {"token": "there", "pos": "PRON"}, {"token": "is", "pos": "VERB"}, {"token": "no", "pos": "DET"}, {"token": "companion", "pos": "NOUN"}, {"token": "quite", "pos": "ADV"}, {"token": "so", "pos": "ADV"}, {"token": "devoted", "pos": "ADJ"}, {"token": "so", "pos": "ADV"}, {"token": "communicative", "pos": "ADJ"}, {"token": "so", "pos": "ADV"}, {"token": "loving", "pos": "ADJ"}, {"token": "and", "pos": "CCONJ"}, {"token": "so", "pos": "ADV"}, {"token": "mesmerizing", "pos": "ADJ"}, {"token": ".", "pos": "PUNCT"}, {"token": "as", "pos": "ADP"}, {"token": "a", "pos": "DET"}, {"token": "rat", "pos": "NOUN"}, {"token": ".", "pos": "PUNCT"}]

Sentence: The first thing to do is to get a good pair of shoes.
Answer: [{"token": "The", "pos": "DET"}, {"token": "first", "pos": "ADJ"}, {"token": "thing", "pos": "NOUN"}, {"token": "to", "pos": "ADP"}, {"token": "do", "pos": "VERB"}, {"token": "is", "pos": "VE

,Token Precision,Token Recall,Token F1,POS Precision,POS Recall,POS F1,valid outputs
base model (plain prompt),0.984,0.949,0.966,0.794,0.767,0.780,20
"instruct, default",0.208,0.136,0.165,0.179,0.117,0.141,18
"instruct, annotator + JSON only + UD rules",0.966,0.654,0.780,0.730,0.494,0.589,18


---

## 9) Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

We used `Llama-3.2-3B-Instruct` (4-bit) on 100 test sentences from the English UD EWT data. The prompt uses the `PromptTemplate` from part 1: it asks the model to split the sentence into tokens and give each token one of the 17 UPOS tags, and it includes 3 example sentences from the training data in JSON format. For the instruct model, we put this prompt into the Llama 3.1 chat template. To read the outputs, we find the first JSON list with a regex, load it with `json` and check it with Pydantic. Outputs that cannot be read count as missing. We tried 4 system prompts and then ran the base model with the same prompt on the first 20 sentences. All runs used greedy decoding on a Colab T4.

The system prompt made the biggest difference (POS F1 from 0.140 to 0.697). On the 20 sentences, the base model was better than the instruct model (POS F1 0.780 vs 0.589).

### Questions

**3) Data**

- 📝❓ What are the fields and their corresponding values in the dataframe?
  - `sentence` is the raw sentence, `tokens` are the UD tokens separated by spaces, `upos` and `xpos` give one tag per token. We added `target`, a list of `{"token", "pos"}` pairs, which is what the model has to predict.
  - There are 1,508 sentences: 100 for testing and 1,408 for training (we only use 3 of them as examples). Test sentences have 12 tokens on average (2 to 64). In 80 of the 100 test sentences, the UD tokens are not the same as splitting the sentence at spaces, mostly because punctuation and clitics like "n't" are separate tokens.

- 📝❓ What is the difference between `upos` and `xpos`?
  - `upos` uses the 17 universal tags that are shared by all UD languages. `xpos` uses a language-specific tag set, here the Penn Treebank tags, which are more detailed (for example `VBZ` vs `VBD` for verb tense, `NN` vs `NNS` for singular and plural).

- 📝❓ What is the distribution of `upos` labels in the `test_data`?
  - The 1,200 test tokens are mostly `NOUN` (219), `PUNCT` (146), `VERB` (124), `ADJ` (111), `PRON` (106), `ADP` (99) and `DET` (88). Some tags are very rare: `INTJ` (6), `SYM` (3) and `X` (1).


**7) System prompt**

- 📝❓ What was the best system prompt you considered?
  - The best one was: "You are an expert linguist who annotates English text following the Universal Dependencies guidelines. Respond only with the JSON list, without any explanation or code fences." followed by a few UD rules (AUX for auxiliaries, PART for "to" and "not", PROPN for names, etc.).

  | System prompt | Valid outputs | Token F1 | POS F1 |
  |---|---|---|---|
  | default (no system message) | 98 | 0.174 | 0.140 |
  | annotator | 63 | 0.183 | 0.139 |
  | annotator + JSON only | 95 | 0.895 | 0.683 |
  | annotator + JSON only + UD rules | 94 | 0.886 | 0.697 |

- 📝❓ Were you able to improve the performance by manipulating the system prompt? Please discuss.
  - Yes, POS F1 went from 0.140 to 0.697. Almost all of this came from telling the model to answer only with the JSON list.
  - With the default prompt, 98 outputs were valid JSON but the precision was only 0.20. The predicted lists had about 9 tokens on average, which is the length of our first example sentence. So the model seems to have tagged the example sentences again, and our parser takes the first list it finds.
  - The UD rules only helped a little (+0.014 POS F1). The most common error is still `AUX` tagged as `VERB` (41 times), and the model also used the old tag `CONJ` instead of `CCONJ` (15 times).
  - Things we would try next: choose the list that matches the input sentence instead of the first one, give the examples as separate chat turns, map `CONJ` to `CCONJ`, use examples with auxiliaries and "to", and retry the outputs that could not be parsed.


**8) Base model vs. instruction-tuned model**

- 📝❓ How do the outputs of the base model differ from those of the instruction-tuned model (output format, number of valid outputs, precision, recall and F1)?
  - Results on the first 20 test sentences, with the same prompt:

  | Model | Valid outputs | Token P / R / F1 | POS P / R / F1 |
  |---|---|---|---|
  | Base model | 20 | 0.984 / 0.949 / 0.966 | 0.794 / 0.767 / 0.780 |
  | Instruct, default | 18 | 0.208 / 0.136 / 0.165 | 0.179 / 0.117 / 0.141 |
  | Instruct, best system prompt | 18 | 0.966 / 0.654 / 0.780 | 0.730 / 0.494 / 0.589 |

  - The base model just continues the pattern from the examples. It writes the JSON for the test sentence and then keeps making up new sentences until it reaches the token limit, so it is slower (2.3 min for 20 sentences). Our parser ignores everything after the first list, so all 20 outputs were valid.
  - The instruct model stops after its answer. With the best system prompt, its precision is close to the base model, but its recall is much lower, because 2 outputs were invalid and it left out more tokens.
  - We did not expect the base model to do better. Our guess is that a few-shot prompt with a fixed format is exactly what a base model is good at.

- 📝❓ In part 1, you finetuned the base model with LoRA on a single task. How is this different from the instruction tuning that turned `Llama-3.2-3B` into `Llama-3.2-3B-Instruct`? Use your results on the out-of-domain names from part 1 in your answer.
  - In part 1, LoRA trained a small adapter (0.75% of the parameters) on one task with one input format. It got 0.86 on two-part names but 0.00 on three-part names, because it mostly ignored the middle name. It learned the format, not the rule.
  - Instruction tuning trains the model on many different tasks written as instructions, and then on human preferences. The model learns to follow instructions in general, which is why it could do POS tagging without being trained for it, and why one sentence in the system prompt changed the results so much. But it does not make the model better at one specific task, as the base model results show.


## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☒ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☒ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:

Claude Opus 5.5 is used for debugging and simplifying code, for graphs, shaping markdown texts (tables, colors etc.), and for understanding certain results. 
